<a href="https://colab.research.google.com/github/Vishwas-Chaudhary/ML-Internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

In [ ]:
import os
import pandas as pd

# Load the starter dataset
data_path = "/content/ML-Internship/data/raw/content_refresh_anonymized.csv"

if not os.path.exists(data_path):
    !git clone https://github.com/Vishwas-Chaudhary/ML-Internship.git

df = pd.read_csv(data_path)

print("Dataset shape:", df.shape)

# ============================================================
# BASELINE RULE
# ============================================================

print("""
BASELINE RULE

I will prioritize content pages that have meaningful search exposure
and appear to have weak CTR relative to their observed search position.

The rule uses two signals:

1. CTR relative to position tier
2. Impression tier

Position is the main opportunity signal. Impression volume is used as
an actionability signal because a review is more useful when a page has
meaningful observed exposure.

This is a directional, decision-support baseline and not a causal claim.
""")

print("""
REASON CODES

CTR_POSITION_WEAK:
CTR is below the typical CTR for the page's position tier.

HIGH_IMPRESSIONS:
The page has relatively high search exposure.

CTR_POSITION_AND_IMPRESSIONS:
Both signals support prioritizing the page.

NO_STRONG_SIGNAL:
Neither signal is strong enough to prioritize the page.
""")

# ============================================================
# SIGNAL 1: CTR VS POSITION TIER
# ============================================================

print("\n" + "=" * 60)
print("SIGNAL 1: CTR BY POSITION TIER")
print("=" * 60)

position_table = (
    df.groupby("position_tier", dropna=False)
      .agg(
          n=("content_id", "size"),
          mean_ctr=("ctr", "mean"),
          median_ctr=("ctr", "median"),
          mean_position=("avg_position", "mean")
      )
      .sort_values("mean_position")
      .reset_index()
)

print(position_table)

print("""
VERDICT: CONFIRMED

Observed CTR changes directionally across position tiers, with higher
CTR generally occurring at better search positions. This supports using
position-adjusted CTR as the main opportunity signal.
""")

# ============================================================
# SIGNAL 2: IMPRESSION TIER
# ============================================================

print("\n" + "=" * 60)
print("SIGNAL 2: IMPRESSION TIER")
print("=" * 60)

impression_table = (
    df.groupby("impression_tier", dropna=False)
      .agg(
          n=("content_id", "size"),
          mean_ctr=("ctr", "mean"),
          median_ctr=("ctr", "median"),
          mean_impressions=("impressions_90d", "mean")
      )
      .sort_values("mean_impressions", ascending=False)
      .reset_index()
)

print(impression_table)

print("""
VERDICT: MIXED

Observed impression tiers clearly represent different exposure levels,
but their relationship with CTR is not consistently directional.
Therefore, impression volume is used as an actionability/context signal,
not as direct evidence of CTR weakness.
""")

# ============================================================
# BUCKET COUNTS
# ============================================================

print("\nPOSITION TIER COUNTS")
print(df["position_tier"].value_counts(dropna=False))

print("\nIMPRESSION TIER COUNTS")
print(df["impression_tier"].value_counts(dropna=False))

Dataset shape: (30000, 44)

BASELINE RULE

I will prioritize content pages that have meaningful search exposure
and appear to have weak CTR relative to their observed search position.

The rule uses two signals:

1. CTR relative to position tier
2. Impression tier

Position is the main opportunity signal. Impression volume is used as
an actionability signal because a review is more useful when a page has
meaningful observed exposure.

This is a directional, decision-support baseline and not a causal claim.


REASON CODES

CTR_POSITION_WEAK:
CTR is below the typical CTR for the page's position tier.

HIGH_IMPRESSIONS:
The page has relatively high search exposure.

CTR_POSITION_AND_IMPRESSIONS:
Both signals support prioritizing the page.

NO_STRONG_SIGNAL:
Neither signal is strong enough to prioritize the page.


SIGNAL 1: CTR BY POSITION TIER
  position_tier      n  mean_ctr  median_ctr  mean_position
0         top_3   2321  1.483611        0.00       1.010728
1        page_1  11814  0.

## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

In [ ]:
import os
import pandas as pd
import numpy as np

# ============================================================
# BUILD BASELINE SCORE
# ============================================================

baseline = df.copy()

# Score based on two observed signals:
# 1. Worse position = higher review priority
# 2. Higher impressions = more actionable exposure

# Position score
position_score = {
    "top_3": 0,
    "page_1": 1,
    "striking": 2,
    "page_3_5": 3,
    "deep": 4
}

baseline["position_score"] = (
    baseline["position_tier"]
    .map(position_score)
    .fillna(0)
)

# Impression score
impression_score = {
    "low": 0,
    "moderate": 1,
    "good": 2,
    "excellent": 3
}

baseline["impression_score"] = (
    baseline["impression_tier"]
    .map(impression_score)
    .fillna(0)
)

# Normalize CTR within the observed position tier.
# Lower CTR than the typical value for the same position
# receives more review priority.

position_ctr_median = (
    baseline.groupby("position_tier")["ctr"]
    .transform("median")
)

baseline["ctr_gap"] = (
    position_ctr_median - baseline["ctr"]
).clip(lower=0)

# Final baseline score
baseline["score"] = (
    baseline["position_score"] * 2
    + baseline["impression_score"]
    + baseline["ctr_gap"]
)

# ============================================================
# REASON CODE
# ============================================================

def get_reason(row):

    weak_ctr = row["ctr_gap"] > 0
    high_impressions = row["impression_score"] >= 2

    if weak_ctr and high_impressions:
        return "CTR_POSITION_AND_IMPRESSIONS"

    elif weak_ctr:
        return "CTR_POSITION_WEAK"

    elif high_impressions:
        return "HIGH_IMPRESSIONS"

    else:
        return "NO_STRONG_SIGNAL"


baseline["reason_code"] = baseline.apply(get_reason, axis=1)

# ============================================================
# ACTION LABEL
# ============================================================

baseline["action"] = np.where(
    baseline["score"] > 0,
    "REVIEW",
    "NO_ACTION"
)

# ============================================================
# RANK THE QUEUE
# ============================================================

baseline = baseline.sort_values(
    by="score",
    ascending=False
).reset_index(drop=True)

baseline["rank"] = baseline.index + 1

# ============================================================
# KEEP ONLY USEFUL OUTPUT COLUMNS
# ============================================================

queue = baseline[
    [
        "rank",
        "content_id",
        "score",
        "reason_code",
        "action",
        "position_tier",
        "impression_tier",
        "ctr",
        "avg_position",
        "impressions_90d"
    ]
].copy()

print("Ranked queue created.")
print("Rows:", len(queue))

print("\nTop 20:")
print(queue.head(20))

# ============================================================
# WRITE CSV
# ============================================================

output_dir = "/content/ML-Internship/work/outputs"
os.makedirs(output_dir, exist_ok=True)

output_path = os.path.join(
    output_dir,
    "baseline_action_score.csv"
)

queue.to_csv(output_path, index=False)

print("\nCSV written to:")
print(output_path)

print("\nTop 10 actions:")
print(queue.head(10)[
    ["rank", "score", "reason_code", "action"]
])

Ranked queue created.
Rows: 30000

Top 20:
    rank            content_id  score       reason_code  action position_tier  \
0      1  content_fb66dd8f4629   11.0  HIGH_IMPRESSIONS  REVIEW          deep   
1      2  content_109f8f7c9d39   11.0  HIGH_IMPRESSIONS  REVIEW          deep   
2      3  content_a023517539fe   11.0  HIGH_IMPRESSIONS  REVIEW          deep   
3      4  content_62abc4bd66be   11.0  HIGH_IMPRESSIONS  REVIEW          deep   
4      5  content_bcf8e8e2280d   11.0  HIGH_IMPRESSIONS  REVIEW          deep   
5      6  content_d376881f5bd0   10.0  HIGH_IMPRESSIONS  REVIEW          deep   
6      7  content_af2b73769867   10.0  HIGH_IMPRESSIONS  REVIEW          deep   
7      8  content_878a1b63eb1b   10.0  HIGH_IMPRESSIONS  REVIEW          deep   
8      9  content_cb5611d87825   10.0  HIGH_IMPRESSIONS  REVIEW          deep   
9     10  content_20073d619610   10.0  HIGH_IMPRESSIONS  REVIEW          deep   
10    11  content_6f3297163cca   10.0  HIGH_IMPRESSIONS  REVIEW   

## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

In [ ]:
# ============================================================
# SECTION 3: TOP-20 REVIEW
# ============================================================

top20 = baseline.head(20).copy()

def confidence_note(row):
    if row["reason_code"] == "CTR_POSITION_AND_IMPRESSIONS":
        return "Higher confidence because both observed signals support review."
    elif row["reason_code"] == "HIGH_IMPRESSIONS":
        return "Moderate confidence because exposure is high, but CTR weakness is not confirmed."
    elif row["reason_code"] == "CTR_POSITION_WEAK":
        return "Moderate confidence because CTR is below the observed position-tier median."
    else:
        return "Low confidence because there is no strong supporting signal."

def wrong_reason(row):
    if row["reason_code"] == "HIGH_IMPRESSIONS":
        return (
            "Could be wrong if high impressions are not actionable or "
            "the low position is expected for this page."
        )
    elif row["reason_code"] == "CTR_POSITION_AND_IMPRESSIONS":
        return (
            "Could be wrong if the CTR difference is caused by query mix, "
            "intent, or other factors not represented in the rule."
        )
    elif row["reason_code"] == "CTR_POSITION_WEAK":
        return (
            "Could be wrong if the low CTR is expected for the page's "
            "search intent or query mix."
        )
    else:
        return "Could be wrong if the available signals do not reflect the actual review opportunity."

top20["confidence_note"] = top20.apply(
    confidence_note,
    axis=1
)

top20["what_would_make_it_wrong"] = top20.apply(
    wrong_reason,
    axis=1
)

top20_review = top20[
    [
        "rank",
        "content_id",
        "action",
        "reason_code",
        "score",
        "position_tier",
        "impression_tier",
        "ctr",
        "avg_position",
        "impressions_90d",
        "confidence_note",
        "what_would_make_it_wrong"
    ]
].copy()

print("TOP-20 REVIEW")
print("=" * 80)

for _, row in top20_review.iterrows():

    print(f"""
Rank {int(row['rank'])}
Content: {row['content_id']}
Action: {row['action']}
Reason code: {row['reason_code']}
Score: {row['score']:.2f}
Position tier: {row['position_tier']}
Impression tier: {row['impression_tier']}
CTR: {row['ctr']:.4f}
Average position: {row['avg_position']:.2f}
Impressions: {int(row['impressions_90d'])}

Confidence: {row['confidence_note']}

What would make it wrong:
{row['what_would_make_it_wrong']}
""")

print("=" * 80)
print("Top-20 review completed.")

TOP-20 REVIEW

Rank 1
Content: content_fb66dd8f4629
Action: REVIEW
Reason code: HIGH_IMPRESSIONS
Score: 11.00
Position tier: deep
Impression tier: excellent
CTR: 0.0900
Average position: 56.50
Impressions: 32518

Confidence: Moderate confidence because exposure is high, but CTR weakness is not confirmed.

What would make it wrong:
Could be wrong if high impressions are not actionable or the low position is expected for this page.


Rank 2
Content: content_109f8f7c9d39
Action: REVIEW
Reason code: HIGH_IMPRESSIONS
Score: 11.00
Position tier: deep
Impression tier: excellent
CTR: 0.0100
Average position: 54.40
Impressions: 90476

Confidence: Moderate confidence because exposure is high, but CTR weakness is not confirmed.

What would make it wrong:
Could be wrong if high impressions are not actionable or the low position is expected for this page.


Rank 3
Content: content_a023517539fe
Action: REVIEW
Reason code: HIGH_IMPRESSIONS
Score: 11.00
Position tier: deep
Impression tier: excellent
C

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

In [ ]:
# ============================================================
# SECTION 4: WEAK PICKS + LEAKAGE CHECK
# ============================================================

print("WEAK PICKS + LEAKAGE CHECK")
print("=" * 80)

# ------------------------------------------------------------
# 1. Identify potentially weak picks
# ------------------------------------------------------------

weak_picks = top20[
    (top20["reason_code"] == "HIGH_IMPRESSIONS") &
    (top20["ctr"] > top20.groupby("position_tier")["ctr"].transform("median"))
].copy()

print("\nPotentially weak picks:")
print(weak_picks[
    [
        "rank",
        "content_id",
        "score",
        "reason_code",
        "position_tier",
        "impression_tier",
        "ctr",
        "avg_position",
        "impressions_90d"
    ]
])

print("""
WHY THESE MAY BE WEAK:

Some top-ranked pages are selected mainly because they have high
impression volume and a deep position. High exposure alone does not
prove that a page has a CTR improvement opportunity.

A page may also have a CTR above the median for its position tier.
Those cases are weaker candidates for review and should be treated
with caution.
""")

# ------------------------------------------------------------
# 2. Check for future-window / leakage columns
# ------------------------------------------------------------

print("\nLEAKAGE CHECK")
print("-" * 80)

# Columns that would clearly represent future outcomes
future_keywords = [
    "future",
    "april",
    "may",
    "june",
    "label",
    "outcome",
    "target"
]

possible_leaks = [
    col for col in baseline.columns
    if any(word in col.lower() for word in future_keywords)
]

print("Possible future/label-derived columns in baseline:")
print(possible_leaks)

# ------------------------------------------------------------
# 3. Confirm columns actually used by the score
# ------------------------------------------------------------

score_inputs = [
    "position_tier",
    "impression_tier",
    "ctr",
    "avg_position",
    "impressions_90d"
]

print("\nActual score inputs:")
print(score_inputs)

leaked_score_inputs = [
    col for col in score_inputs
    if any(word in col.lower() for word in future_keywords)
]

print("\nFuture/label-derived score inputs:")
print(leaked_score_inputs)

# ------------------------------------------------------------
# 4. Final verdict
# ------------------------------------------------------------

if len(leaked_score_inputs) == 0:
    print("""
LEAKAGE VERDICT: PASS

The baseline score uses only observed page performance signals.
No future-window or label-derived variable is used as a scoring input.
""")
else:
    print("""
LEAKAGE VERDICT: FAIL

A future or label-derived variable was included in the score.
Remove it before using the baseline.
""")

print("""
LIMITATION:

The baseline is a simple hand-written decision-support rule.
It does not establish causality and may prioritize pages that have
high exposure but limited improvement opportunity. The top-20 review
shows why human review is still needed.
""")

WEAK PICKS + LEAKAGE CHECK

Potentially weak picks:
    rank            content_id  score       reason_code position_tier  \
0      1  content_fb66dd8f4629   11.0  HIGH_IMPRESSIONS          deep   
3      4  content_62abc4bd66be   11.0  HIGH_IMPRESSIONS          deep   
7      8  content_878a1b63eb1b   10.0  HIGH_IMPRESSIONS          deep   
8      9  content_cb5611d87825   10.0  HIGH_IMPRESSIONS          deep   
9     10  content_20073d619610   10.0  HIGH_IMPRESSIONS          deep   
10    11  content_6f3297163cca   10.0  HIGH_IMPRESSIONS          deep   
13    14  content_38ecd424bbc6   10.0  HIGH_IMPRESSIONS          deep   
15    16  content_94d1f783deed   10.0  HIGH_IMPRESSIONS          deep   
16    17  content_3250919d61a0   10.0  HIGH_IMPRESSIONS          deep   

   impression_tier   ctr  avg_position  impressions_90d  
0        excellent  0.09          56.5            32518  
3        excellent  0.09          68.5            31364  
7             good  0.22          71.1     

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.